# Patent Litigation Data Pipeline

PACER (cases.csv / patents.csv / names.csv) + USPTO PatentsView 데이터로
MANTF 모델 입력 파일 생성

| Step | 내용 |
|------|------|
| 1 | 데이터 로드 |
| 2 | 침해소송 필터링 |
| 3 | 특허번호 정규화 |
| 4 | 원고/피고 분류 (Counter 계열 포함) |
| 5 | 케이스별 집계 → 1케이스 1행 |
| 6 | 기업명 정규화 |
| 7 | Assignee 매칭 → 원고-특허 연결 |
| 8 | Train / Val / Test 분할 + Negative Sampling |
| 9 | 인덱스 파일 생성 |
| 10 | 기업 소송 그래프 생성 |
| 11 | 특허 BERT 임베딩 |
| 12 | 특허 DeepWalk 임베딩 |

In [1]:
import re
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

DATA_DIR   = Path('.')          # cases.csv, patents.csv, names.csv 위치
USPTO_DIR  = Path('.')          # g_patent.tsv 등 USPTO 파일 위치
OUTPUT_DIR = Path('../data')

(OUTPUT_DIR / 'samples').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'features').mkdir(parents=True, exist_ok=True)
print('준비 완료')

준비 완료


## Step 1. 데이터 로드

In [2]:
cases   = pd.read_csv(DATA_DIR / 'cases.csv',   low_memory=False)
patents = pd.read_csv(DATA_DIR / 'patents.csv',  low_memory=False)
names   = pd.read_csv(DATA_DIR / 'names.csv',    low_memory=False)

print(f'cases:   {len(cases):,}행  columns={list(cases.columns)}')
print(f'patents: {len(patents):,}행  columns={list(patents.columns)}')
print(f'names:   {len(names):,}행  columns={list(names.columns)}')

cases:   81,350행  columns=['case_row_id', 'case_number', 'pacer_id', 'court_name', 'district_id', 'case_name', 'assigned_to', 'referred_to', 'case_cause', 'jurisdictional_basis', 'date_filed', 'date_closed', 'date_last_filed', 'jury_demand', 'demand', 'lead_case', 'related_case', 'settlement', 'case_number_raw', 'case_type_1', 'case_type_2', 'case_type_3', 'case_type_note']
patents: 122,252행  columns=['case_row_id', 'pacer_id', 'case_number', 'district_id', 'nos', 'date_filed', 'case_name', 'case_type_1', 'case_type_2', 'case_type_3', 'case_type_note', 'patent', 'patent_doc_type']
names:   745,235행  columns=['case_row_id', 'case_number', 'party_row_count', 'party_type', 'name_row_count', 'name']


## Step 2. 침해소송 필터링

`case_type_1 == 1.0` = Patent Infringement (non-DJ)

In [3]:
inf_cases = cases[cases['case_type_1'] == 1.0][['case_row_id', 'date_filed']].copy()
inf_cases['date_filed'] = pd.to_datetime(inf_cases['date_filed'], errors='coerce')

print(f'전체 케이스:     {len(cases):,}건')
print(f'침해소송 케이스: {len(inf_cases):,}건')

전체 케이스:     81,350건
침해소송 케이스: 46,907건


In [4]:
# 날짜 범위 단계별 확인 (우리가 직접 연도 필터링한 건 없음 — 자연 탈락 원인 파악용)
inf_all = cases[cases['case_type_1'] == 1.0].copy()
inf_all['date_filed'] = pd.to_datetime(inf_all['date_filed'], errors='coerce')

has_pat   = inf_all[inf_all['case_row_id'].isin(lit_patents['case_row_id'])]
has_party = inf_all[inf_all['case_row_id'].isin(lit_names['case_row_id'])]

print('=== 날짜 범위 단계별 확인 ===')
print(f'cases.csv 전체:          {cases["date_filed"].min()} ~ {cases["date_filed"].max()}')
print(f'침해소송(case_type_1=1): {inf_all["date_filed"].min()} ~ {inf_all["date_filed"].max()}')
print(f'+ 유효 특허 있는 케이스: {has_pat["date_filed"].min()} ~ {has_pat["date_filed"].max()}  ({len(has_pat):,}건)')
print(f'+ 원고/피고 있는 케이스: {has_party["date_filed"].min()} ~ {has_party["date_filed"].max()}  ({len(has_party):,}건)')
print(f'최종 case_table:         {case_table["date_filed"].min()} ~ {case_table["date_filed"].max()}  ({len(case_table):,}건)')

NameError: name 'lit_patents' is not defined

## Step 3. 특허번호 정규화

- `patent_doc_type == 'Patent'` 만 사용 (Application 제외)
- 앞자리 0 제거, 영문 prefix 유지 (대문자)
- 예: `06457274` → `6457274`, `D0382190` → `D382190`, `RE037721` → `RE37721`

In [5]:
def normalize_patent(s):
    if pd.isna(s):
        return None
    s = str(s).strip().upper()
    m = re.match(r'^([A-Z]*)(\d+)$', s)
    if not m:
        return None
    prefix, digits = m.group(1), m.group(2)
    return prefix + digits.lstrip('0')

lit_patents = patents[
    (patents['case_row_id'].isin(inf_cases['case_row_id'])) &
    (patents['patent_doc_type'].str.strip() == 'Patent')
].copy()
lit_patents['patent_norm'] = lit_patents['patent'].astype(str).apply(normalize_patent)
lit_patents = lit_patents[lit_patents['patent_norm'].notna()]

print(f'분쟁 특허 레코드: {len(lit_patents):,}건')
print(f'고유 분쟁 특허:   {lit_patents["patent_norm"].nunique():,}개')

분쟁 특허 레코드: 103,711건
고유 분쟁 특허:   38,207개


In [6]:
# 단계별 필터링 현황
s1 = patents
s2 = patents[patents['case_row_id'].isin(inf_cases['case_row_id'])]
s3 = s2[s2['patent_doc_type'].str.strip() == 'Patent']
s4_tmp = s3.copy()
s4_tmp['patent_norm'] = s3['patent'].astype(str).apply(normalize_patent)
s4 = s4_tmp[s4_tmp['patent_norm'].notna()]

print('=== 특허 필터링 단계별 현황 ===')
print(f'patents.csv 전체:                        {len(s1):,}행')
print(f'  └ 침해소송 케이스에 속하는 행:          {len(s2):,}행  (케이스 {s2["case_row_id"].nunique():,}건 포함)')
print(f'      └ patent_doc_type=="Patent":        {len(s3):,}행')
print(f'          └ 특허번호 정규화 성공:          {len(s4):,}행  ← lit_patents')
print()
print(f'침해소송 케이스 전체:                     {len(inf_cases):,}건')
print(f'  └ 특허가 1개 이상 있는 케이스:          {s4["case_row_id"].nunique():,}건')
print(f'  └ 특허가 없어서 이후 탈락할 케이스:     {len(inf_cases) - s4["case_row_id"].nunique():,}건')
print()
print(f'고유 분쟁 특허번호 (BERT/DeepWalk 대상):  {s4["patent_norm"].nunique():,}개')
print()
print('--- patent_doc_type 종류 (침해소송 케이스 내) ---')
print(s2['patent_doc_type'].value_counts())

=== 특허 필터링 단계별 현황 ===
patents.csv 전체:                        122,252행
  └ 침해소송 케이스에 속하는 행:          103,795행  (케이스 46,907건 포함)
      └ patent_doc_type=="Patent":        103,711행
          └ 특허번호 정규화 성공:          103,711행  ← lit_patents

침해소송 케이스 전체:                     46,907건
  └ 특허가 1개 이상 있는 케이스:          46,834건
  └ 특허가 없어서 이후 탈락할 케이스:     73건

고유 분쟁 특허번호 (BERT/DeepWalk 대상):  38,207개

--- patent_doc_type 종류 (침해소송 케이스 내) ---
patent_doc_type
Patent                   103711
Application                  20
Published Application         9
Name: count, dtype: int64


## Step 4. 원고/피고 분류

| side | 해당 party_type | 설명 |
|------|----------------|------|
| P | Plaintiff 계열 | 원고 |
| D | Defendant 계열 | 피고 |
| CP | Counter Defendant 계열 | 반소 당하는 쪽 = 원래 원고 |
| CD | Counter Claimant / Counter Plaintiff 계열 | 반소 제기하는 쪽 = 원래 피고 |

위 4개 집합에 없는 타입(Intervenor, Third Party 등)은 제외

In [25]:
PLAINTIFF_TYPES = {
    'Plaintiff', 'plaintiff',
    'Plaintiff Consolidated', 'Consolidated Plaintiff',
    'Consolidated Filer Plaintiff', 'Consol Plaintiff',
    'Plaintiff - Defaulted', 'Plaintiff, Counter Defendant',
}
DEFENDANT_TYPES = {
    'Defendant', 'defendant',
    'Consol Defendant', 'Defendant Consolidated', 'Consolidated Defendant',
}
COUNTER_AS_PLAINTIFF = {
    'Counter Defendant', 'Counter-defendant', 'Counter-Defendant',
    'counter-defendant', 'Counter Defendant Consolidated',
    'Consol Counter Defendant', 'Consol CounterDft',
    'Counterclaim Defendant', 'Consolidated Counter Defendant',
}
COUNTER_AS_DEFENDANT = {
    'Counter Claimant', 'Counter-claimant', 'Counter-Claimant',
    'counter-claimant', 'Counter Claimant Consolidated',
    'Consol Counter Claimant', 'Consol CounterCla',
    'Counter Plaintiff', 'Counter-Plaintiff', 'Counterclaim Plaintiff',
    'Consol Counter Plaintiff',
}

def classify_side(pt):
    if pt in PLAINTIFF_TYPES:       return 'P'
    if pt in DEFENDANT_TYPES:       return 'D'
    if pt in COUNTER_AS_PLAINTIFF:  return 'CP'
    if pt in COUNTER_AS_DEFENDANT:  return 'CD'
    return None

lit_names = names[names['case_row_id'].isin(inf_cases['case_row_id'])].copy()
lit_names['side'] = lit_names['party_type'].apply(classify_side)

print('=== 분류 제외된 party_type (상위 20) ===')
print(lit_names[lit_names['side'].isna()]['party_type'].value_counts().head(20))

lit_names = lit_names[lit_names['side'].notna() & lit_names['name'].notna()]
print(f'\n분류된 레코드: {len(lit_names):,}건')
print(lit_names['side'].value_counts())

=== 분류 제외된 party_type (상위 20) ===
party_type
Mediator                 3861
Notice Only              3368
Movant                   1449
ThirdParty Defendant     1418
ThirdParty Plaintiff     1196
Technical Advisor        1104
Interested Party          763
Intervenor                714
Special Master            674
Cross Defendant           659
Cross Claimant            500
Miscellaneous             419
Third Party Defendant     361
Respondent                346
Intervenor Defendant      290
Mediator (ADR Panel)      236
MDL - Notice Only         196
Intervenor Plaintiff      183
Conflict Check Party      152
3rd party defendant       150
Name: count, dtype: int64

분류된 레코드: 488,425건
side
D     222983
CD    105747
P      91289
CP     68406
Name: count, dtype: int64


## Step 5. 케이스별 집계 → 1케이스 1행

행 순서대로 읽으면서 이름이 처음 등장할 때 side로 확정.  
같은 이름이 뒤에 다른 타입으로 재등장해도 skip.

In [26]:
def aggregate_parties(group):
    """
    행 순서대로 처리:
    - 이름이 처음 등장하면 해당 side로 배정
    - 이미 배정된 이름은 뒤에 어떤 타입으로 재등장해도 skip
    """
    classified = {}  # name → 'P' or 'D'

    for _, row in group.iterrows():
        name = row['name']
        side = row['side']
        if name in classified:
            continue
        if side in ('P', 'CP'):
            classified[name] = 'P'
        elif side in ('D', 'CD'):
            classified[name] = 'D'

    plaintiffs = sorted(n for n, s in classified.items() if s == 'P')
    defendants = sorted(n for n, s in classified.items() if s == 'D')

    return pd.Series({
        'plaintiffs': ' | '.join(plaintiffs) if plaintiffs else None,
        'defendants': ' | '.join(defendants) if defendants else None,
    })

print('원고/피고 집계 중...')
parties = (
    lit_names
    .groupby('case_row_id')
    .apply(aggregate_parties, include_groups=False)
    .reset_index()
)
print(f'집계 완료: {len(parties):,}건')

원고/피고 집계 중...
집계 완료: 46,294건


In [27]:
# 특허 집계
patents_agg = (
    lit_patents
    .groupby('case_row_id')['patent_norm']
    .apply(lambda x: ' | '.join(sorted(x.dropna().unique())))
    .rename('patents')
    .reset_index()
)

# 최종 조인
case_table = (
    inf_cases
    .merge(parties,     on='case_row_id', how='inner')
    .merge(patents_agg, on='case_row_id', how='inner')
    .dropna(subset=['plaintiffs', 'defendants', 'patents'])
    .reset_index(drop=True)
)

# 자기소송 확인 및 제거 (원고∩피고 이름이 겹치는 케이스)
def has_overlap(row):
    return bool(set(row['plaintiffs'].split(' | ')) & set(row['defendants'].split(' | ')))

overlap_cases = case_table[case_table.apply(has_overlap, axis=1)]
print(f'자기소송 케이스: {len(overlap_cases)}건')
if len(overlap_cases) > 0:
    print('샘플:')
    print(overlap_cases[['case_row_id', 'plaintiffs', 'defendants']].head(5).to_string())

case_table = case_table[~case_table.apply(has_overlap, axis=1)].reset_index(drop=True)

print(f'\n최종 케이스: {len(case_table):,}건')
print(f'date_filed:  {case_table["date_filed"].min()} ~ {case_table["date_filed"].max()}')
case_table.head(3)

자기소송 케이스: 0건

최종 케이스: 46,153건
date_filed:  2001-05-15 00:00:00 ~ 2016-12-30 00:00:00


,case_row_id,date_filed,plaintiffs,defendants,patents
0,727,2003-01-30,"Magnum Research, Inc",Advanced Barrel Systems | Michael Degerness | ...,5600912 | 6457274
1,728,2003-02-03,"Crestliner Inc | Crestliner, Inc.",Alumacraft Boat Co,5108321
2,729,2003-02-04,Taylor Corporation,"AS Hospitality, Inc.",6153278


In [28]:
# 기초 통계
print('원고 수 분포 (상위 10):')
print(case_table['plaintiffs'].str.split(' | ').apply(len).value_counts().sort_index().head(10))
print('\n피고 수 분포 (상위 10):')
print(case_table['defendants'].str.split(' | ').apply(len).value_counts().sort_index().head(10))
print('\n특허 수 분포 (상위 10):')
print(case_table['patents'].str.split(' | ').apply(len).value_counts().sort_index().head(10))

원고 수 분포 (상위 10):
plaintiffs
1        76
2      6651
3     10618
4      6042
5      2669
6      2181
7      4590
8      1769
9      2522
10     1152
Name: count, dtype: int64

피고 수 분포 (상위 10):
defendants
1      191
2     5750
3     7237
4     3575
5     3069
6     1952
7     3130
8     1913
9     2047
10    1378
Name: count, dtype: int64

특허 수 분포 (상위 10):
patents
1     25251
3      9117
5      4533
7      2629
9      1631
11      985
13      604
15      316
17      260
19      167
Name: count, dtype: int64


## Step 6. 기업명 정규화

같은 회사가 다르게 표기된 경우를 통일:
- `Apple Inc.` / `Apple, Inc.` / `APPLE INC` → `apple inc`
- 법인 suffix 통일 (incorporated→inc, corporation→corp 등)
- 구두점·DBA 표기 제거, 소문자화

In [29]:
# Cell 32
NOISE_PATTERNS = [
    (r'\bet\s+al\.?\s*$',                  ''),
    (r',?\s*doing\s+business\s+as.*$',      ''),
    (r',?\s*d/?b/?a.*$',                    ''),
    (r',?\s*as\s+partners.*$',              ''),
    (r'\bincorp(?:orated)?\b',              'inc'),
    (r'\bcorporation\b',                    'corp'),
    (r'\blimited\b',                        'ltd'),
    (r'\bcompany\b',                        'co'),
    (r'[,\.\-\(\)\/\\&\'"]+',             ' '),
    (r'\s{2,}',                             ' '),
]



def normalize_company(name):
    if pd.isna(name):
        return None
    s = str(name).strip().lower()
    if s in ('doing business as', ''):
        return None
    for pattern, repl in NOISE_PATTERNS:
        s = re.sub(pattern, repl, s, flags=re.IGNORECASE)
    s = s.strip()
    if not s:
        return None
    # "Does" 플레이스홀더 피고 필터 (e.g., "Does 1-10, Inclusive")
    _does_stripped = re.sub(r'\b(?:does?|through|thru|to|inclusive)\b|\d+', '', s).strip()
    if not _does_stripped and re.search(r'\bdoes?\b', s):
        return None
    if re.fullmatch(r'(?:john|jane)\s+does?', s):
        return None

    return s

# 동작 확인
test_names = [
    'Apple Inc.', 'Apple, Inc.', 'APPLE INC',
    'Wang Laboratories Inc.,', 'Robert M. Spargo, Sr, as partners',
    'IBM', '3M', 'LG Electronics Inc.',
    'Does 1-10, Inclusive', 'Does 1 through 20', 'John Doe',
    'a Florida corporation', 'a Delaware corporation', 'a New York corporation',
    'a computer corp',
]
for n in test_names:
    print(f'  {n!r:40s} → {normalize_company(n)!r}')

  'Apple Inc.'                             → 'apple inc'
  'Apple, Inc.'                            → 'apple inc'
  'APPLE INC'                              → 'apple inc'
  'Wang Laboratories Inc.,'                → 'wang laboratories inc'
  'Robert M. Spargo, Sr, as partners'      → 'robert m spargo sr'
  'IBM'                                    → 'ibm'
  '3M'                                     → '3m'
  'LG Electronics Inc.'                    → 'lg electronics inc'
  'Does 1-10, Inclusive'                   → None
  'Does 1 through 20'                      → None
  'John Doe'                               → None
  'a Florida corporation'                  → 'a florida corp'
  'a Delaware corporation'                 → 'a delaware corp'
  'a New York corporation'                 → 'a new york corp'
  'a computer corp'                        → 'a computer corp'


In [ ]:
# Cell 32-diag: does / a-state-corp 필터링 사전 진단

def _new_filter_type(raw_name):
    # 새로 추가된 필터 규칙에만 해당하는지 분류 (None = 필터 안됨)
    if pd.isna(raw_name):
        return None
    s = str(raw_name).strip().lower()
    for pat, repl in NOISE_PATTERNS:
        s = re.sub(pat, repl, s, flags=re.IGNORECASE)
    s = s.strip()
    if not s:
        return None
    does_stripped = re.sub(r'\b(?:does?|through|thru|to|inclusive)\b|\d+', '', s).strip()
    if not does_stripped and re.search(r'\bdoes?\b', s):
        return 'does'
    if re.fullmatch(r'(?:john|jane)\s+does?', s):
        return 'does'
    m = re.fullmatch(r'a\s+(.+?)\s+corp', s)
    if m and m.group(1) in _US_STATES:
        return 'a_state_corp'
    return None

all_names_raw = pd.Series([
    raw
    for col in ['plaintiffs', 'defendants']
    for cell_val in case_table[col].dropna()
    for raw in cell_val.split(' | ')
])

filter_type = all_names_raw.apply(_new_filter_type)
mask = filter_type.notna()

does_cnt  = (filter_type == 'does').sum()
state_cnt = (filter_type == 'a_state_corp').sum()
print(f'새 필터로 제거될 기업명 항목 (등장 횟수 기준):')
print(f'  Does 플레이스홀더 : {does_cnt:,}건')
print(f'  a <state> corp   : {state_cnt:,}건')
print(f'  합계              : {mask.sum():,}건')
print()

sample_df = (
    pd.DataFrame({'원본': all_names_raw[mask], '유형': filter_type[mask]})
    .groupby(['원본', '유형'], sort=False)
    .size()
    .rename('빈도')
    .reset_index()
    .sort_values('빈도', ascending=False)
    .head(20)
    .reset_index(drop=True) 
)
sample_df

새 필터로 제거될 기업명 항목 (등장 횟수 기준):
  Does 플레이스홀더 : 2,374건
  a <state> corp   : 7,771건
  합계              : 10,145건



,원본,유형,빈도
0,a Delaware corporation,a_state_corp,1624
1,a California corporation,a_state_corp,1444
2,Does,does,687
3,a Delaware Corporation,a_state_corp,612
4,a California Corporation,a_state_corp,366
5,a Florida corporation,a_state_corp,303
6,a Washington corporation,a_state_corp,296
7,DOES,does,238
8,DOES 1 through 10 inclusive,does,230
9,DOES 1 THROUGH 10 INCLUSIVE,does,227


In [16]:
query = "florida corporation" \
 # ← 여기에 찾고 싶은 이름 입력

mask = (
    case_table['plaintiffs'].str.contains(query, case=False, na=False) |
    case_table['defendants'].str.contains(query, case=False, na=False)
)
case_table[mask][['case_row_id', 'plaintiffs', 'defendants', 'patents']]
query = "Pascall Coinex"  # ← 여기에 찾고 싶은 이름 입력

mask = (
    case_table['plaintiffs'].str.contains(query, case=False, na=False) |
    case_table['defendants'].str.contains(query, case=False, na=False)
)
case_table[mask][['case_row_id', 'plaintiffs', 'defendants', 'patents']]

,case_row_id,plaintiffs,defendants,patents


In [30]:
def normalize_party_list(s):
    """'A | B | C' 형태의 기업명 문자열을 정규화 후 중복 제거"""
    if pd.isna(s):
        return None
    result = sorted(set(
        n for raw in s.split(' | ')
        if (n := normalize_company(raw)) is not None
    ))
    return ' | '.join(result) if result else None

case_table['plaintiffs'] = case_table['plaintiffs'].apply(normalize_party_list)
case_table['defendants'] = case_table['defendants'].apply(normalize_party_list)
case_table = case_table.dropna(subset=['plaintiffs', 'defendants']).reset_index(drop=True)

# 정규화 후 다시 자기소송 제거 (정규화로 이름이 같아진 케이스)
before = len(case_table)
case_table = case_table[~case_table.apply(has_overlap, axis=1)].reset_index(drop=True)

print(f'정규화 후 케이스: {len(case_table):,}건 (추가 {before - len(case_table)}건 제거)')
case_table.head(3)

정규화 후 케이스: 45,842건 (추가 306건 제거)


,case_row_id,date_filed,plaintiffs,defendants,patents
0,727,2003-01-30,magnum research inc,advanced barrel systems | carbonbarrels com | ...,5600912 | 6457274
1,728,2003-02-03,crestliner inc,alumacraft boat co,5108321
2,729,2003-02-04,taylor corp,as hospitality inc,6153278


In [31]:
# 정규화 적용 전 진단 — 308건이 왜 사라지는지 확인
def normalize_party_list_tmp(s):
    if pd.isna(s): return None
    result = sorted(set(n for raw in s.split(' | ') if (n := normalize_company(raw)) is not None))
    return ' | '.join(result) if result else None

tmp = case_table.copy()
tmp['p_norm'] = tmp['plaintiffs'].apply(normalize_party_list_tmp)
tmp['d_norm'] = tmp['defendants'].apply(normalize_party_list_tmp)

# 이유 1: 이름이 전부 None이 되어 원고/피고 자체가 사라지는 케이스
null_drop = tmp[tmp['p_norm'].isna() | tmp['d_norm'].isna()]
print(f'이유 1 — 이름이 None으로 사라진 케이스: {len(null_drop)}건')
if len(null_drop):
    for _, r in null_drop.head(3).iterrows():
        print(f'  plaintiffs: {r["plaintiffs"]!r}  →  {r["p_norm"]!r}')
        print(f'  defendants: {r["defendants"]!r}  →  {r["d_norm"]!r}')
        print()

# 이유 2: 정규화 후 원고∩피고 이름이 겹치는 케이스
valid = tmp.dropna(subset=['p_norm', 'd_norm'])
overlap_after = valid[valid.apply(
    lambda r: bool(set(r['p_norm'].split(' | ')) & set(r['d_norm'].split(' | '))), axis=1
)]
print(f'이유 2 — 정규화 후 원고=피고 된 케이스: {len(overlap_after)}건')
if len(overlap_after):
    for _, r in overlap_after.head(3).iterrows():
        overlap = set(r['p_norm'].split(' | ')) & set(r['d_norm'].split(' | '))
        print(f'  겹치는 이름: {overlap}')
        print(f'  원고: {r["p_norm"]}')
        print(f'  피고: {r["d_norm"]}')
        print()

이유 1 — 이름이 None으로 사라진 케이스: 0건
이유 2 — 정규화 후 원고=피고 된 케이스: 0건


## Step 7. Assignee 매칭 → 원고-특허 연결

`g_assignee_disambiguated.tsv`에서 특허 소유 기업을 가져와
원고 이름과 비교 → 진짜 특허권자(원고) 특정

**매칭 로직**
- assignee 1토큰: 완전일치
- assignee 2토큰 이상: 공통 토큰 2개 이상이면 매칭

In [32]:
litigated_set = set(lit_patents['patent_norm'].dropna().unique())

print('g_assignee_disambiguated.tsv 로딩 중...')
assignee_chunks = []
for chunk in pd.read_csv(
    USPTO_DIR / 'g_assignee_disambiguated.tsv', sep='\t',
    usecols=['patent_id', 'disambig_assignee_organization', 'assignee_sequence'],
    chunksize=500_000
):
    chunk['patent_id'] = chunk['patent_id'].astype(str)
    matched = chunk[chunk['patent_id'].isin(litigated_set)]
    if len(matched):
        assignee_chunks.append(matched)

assignee = pd.concat(assignee_chunks, ignore_index=True)

# 특허당 대표 assignee (sequence 가장 낮은 것)
assignee_primary = (
    assignee.sort_values('assignee_sequence')
    .groupby('patent_id')['disambig_assignee_organization']
    .first()
    .reset_index()
    .rename(columns={'patent_id': 'patent_norm', 'disambig_assignee_organization': 'assignee_name'})
)
assignee_primary['assignee_norm'] = assignee_primary['assignee_name'].apply(normalize_company)

print(f'assignee 확보: {len(assignee_primary):,}건')
assignee_primary.head(3)

g_assignee_disambiguated.tsv 로딩 중...
assignee 확보: 31,023건


,patent_norm,assignee_name,assignee_norm
0,3932328,Johnson & Johnson GmbH,johnson johnson gmbh
1,3932903,"HILL-ROM, INC.",hill rom inc
2,3963061,Drummond Scientific Company,drummond scientific co


In [33]:
def match_assignee_to_plaintiff(plaintiffs_str, assignee_norm):
    if pd.isna(plaintiffs_str) or pd.isna(assignee_norm):
        return ''
    a_tok = set(str(assignee_norm).split())
    for p in plaintiffs_str.split(' | '):
        p_tok = set(str(p).split())
        if len(a_tok) <= 1:
            if a_tok == p_tok:
                return p
        else:
            if len(a_tok & p_tok) >= 2:
                return p
    return ''

# 케이스-특허 단위로 assignee 조인
case_pat = (
    lit_patents[['case_row_id', 'patent_norm']]
    .merge(assignee_primary, on='patent_norm', how='left')
    .merge(case_table[['case_row_id', 'plaintiffs']], on='case_row_id', how='left')
)
case_pat['matched_plaintiff'] = case_pat.apply(
    lambda r: match_assignee_to_plaintiff(r['plaintiffs'], r['assignee_norm']), axis=1
)

match_rate = (case_pat['matched_plaintiff'] != '').mean()
print(f'원고-특허 assignee 매칭률: {match_rate:.1%}')
case_pat.head(5)

원고-특허 assignee 매칭률: 40.6%


,case_row_id,patent_norm,assignee_name,assignee_norm,plaintiffs,matched_plaintiff
0,727,6457274,NaN,NaN,magnum research inc,
1,728,5108321,"Crestliner, Inc.",crestliner inc,crestliner inc,crestliner inc
2,729,6153278,"The Taylor Group, Inc.",the taylor group inc,taylor corp,
3,731,5897576,SurVivaLink Corporation,survivalink corp,NaN,
4,732,5365690,ECOLAB INC.,ecolab inc,ecolab inc,ecolab inc


## Step 8. 트리플 생성 + Train / Val / Test 분할 + Negative Sampling

**트리플 구성**
- 대표 원고: assignee 매칭 성공한 원고 우선, 없으면 첫 번째
- 대표 피고: 첫 번째
- 특허: 케이스에 연결된 모든 특허 → 특허마다 트리플 1개

**분할**: `date_filed` 기준 시간순 80/10/10

**Negative Sampling**
- neg_type 0: 원고 교체 / 1: 피고 교체 / 2: 특허 교체
- 세 타입 동일 수 생성 (학습 시 0·1을 50% drop → 실질 1:1:2)

In [34]:
# 케이스별 특허-원고 매칭 현황 (case_pat에 이미 plaintiffs 컬럼 있음)
case_pat_stats = case_pat.groupby('case_row_id').agg(
    total_patents   = ('patent_norm', 'count'),
    matched_patents = ('matched_plaintiff', lambda x: (x != '').sum())
).reset_index()
case_pat_stats['match_rate'] = case_pat_stats['matched_patents'] / case_pat_stats['total_patents']

print('=== 케이스별 특허-원고 매칭 현황 ===')
print(case_pat_stats['match_rate'].describe().round(3))
print()
print(f'매칭률 0%    (모든 특허 assignee 못 찾음): {(case_pat_stats["match_rate"] == 0).sum():,}건')
print(f'매칭률 1~99% (일부만 매칭):               {((case_pat_stats["match_rate"] > 0) & (case_pat_stats["match_rate"] < 1)).sum():,}건')
print(f'매칭률 100%  (모든 특허 매칭):            {(case_pat_stats["match_rate"] == 1).sum():,}건')
print()

print('=== 케이스당 특허 수별 평균 매칭률 ===')
case_pat_stats['bucket'] = case_pat_stats['total_patents'].clip(upper=5).map(
    {1:'특허 1개', 2:'특허 2개', 3:'특허 3개', 4:'특허 4개', 5:'특허 5개+'}
)
print(case_pat_stats.groupby('bucket').agg(
    케이스수  = ('case_row_id', 'count'),
    평균매칭률 = ('match_rate', lambda x: f'{x.mean():.1%}')
).to_string())
print()

# 매칭 실패 샘플 — case_pat에 plaintiffs가 이미 있으므로 재merge 불필요
unmatched = case_pat[
    (case_pat['matched_plaintiff'] == '') &
    (case_pat['assignee_norm'].notna()) &
    case_pat['case_row_id'].isin(case_table['case_row_id'])
]
print('=== 매칭 실패 샘플 (assignee는 있으나 원고와 불일치) ===')
for _, r in unmatched.head(5).iterrows():
    print(f'  특허: {r["patent_norm"]}  assignee: {r["assignee_norm"]!r}')
    print(f'  원고: {r["plaintiffs"]!r}')
    print()

=== 케이스별 특허-원고 매칭 현황 ===
count    46834.000
mean         0.372
std          0.462
min          0.000
25%          0.000
50%          0.000
75%          1.000
max          1.000
Name: match_rate, dtype: float64

매칭률 0%    (모든 특허 assignee 못 찾음): 27,112건
매칭률 1~99% (일부만 매칭):               4,567건
매칭률 100%  (모든 특허 매칭):            15,155건

=== 케이스당 특허 수별 평균 매칭률 ===
         케이스수  평균매칭률
bucket              
특허 1개   25678  32.9%
특허 2개    9222  40.5%
특허 3개    4589  43.6%
특허 4개    2665  46.6%
특허 5개+   4680  43.1%

=== 매칭 실패 샘플 (assignee는 있으나 원고와 불일치) ===
  특허: 6153278  assignee: 'the taylor group inc'
  원고: 'taylor corp'

  특허: 4810123  assignee: 'power flo products corp'
  원고: 'diversified dynamics corp'

  특허: 5579919  assignee: 'survivalink corp'
  원고: 'a delaware corp | cardiac science inc'

  특허: 4625275  assignee: 'republic money orders inc'
  원고: 'a minnesota corp | travelers express co inc'

  특허: 6411942  assignee: 'fujitsu ltd'
  원고: 'a delaware corp | firepond inc'



In [35]:
# Step 7-2. Assignment Chain — date_filed 시점 실소유자 추적
# USPTO Patent Assignment 이력으로 소송 제기일 기준 실제 특허 소유자를 찾음

CHAIN_DIR = Path('.')   # assignment_conveyance.csv, assignment.csv, assignee.csv, documentid.csv 위치
OWNERSHIP_TYPES = {'assignment', 'namechg', 'merger', 'employee'}
INTERVAL_CACHE  = Path('../data/ownership_intervals.parquet')

if INTERVAL_CACHE.exists():
    print('소유권 구간 캐시 로드...')
    intervals_df = pd.read_csv(INTERVAL_CACHE, parse_dates=["start_dt", "end_dt"])
else:
    # 1. 실 소유권 이전 rf_id
    print('assignment_conveyance.csv 로드...')
    ac = pd.read_csv(CHAIN_DIR / 'assignment_conveyance.csv', usecols=['rf_id', 'convey_ty'])
    valid_rfids = set(ac.loc[ac['convey_ty'].isin(OWNERSHIP_TYPES), 'rf_id'])
    del ac
    print(f'  소유권 이전 rf_id: {len(valid_rfids):,}개')

    # 2. assignment → record_dt
    print('assignment.csv 로드...')
    asgn = pd.read_csv(CHAIN_DIR / 'assignment.csv',
                       usecols=['rf_id', 'record_dt', 'purge_in'], low_memory=False)
    asgn = asgn[asgn['rf_id'].isin(valid_rfids) & (asgn['purge_in'] == 'N')]
    asgn['record_dt'] = pd.to_datetime(asgn['record_dt'], errors='coerce')
    asgn = asgn.dropna(subset=['record_dt']).drop_duplicates('rf_id')[['rf_id', 'record_dt']]
    valid_rfids2 = set(asgn['rf_id'])
    print(f'  유효 assignment: {len(asgn):,}개')

    # 3. assignee → ee_norm
    print('assignee.csv 로드...')
    ee = pd.read_csv(CHAIN_DIR / 'assignee.csv', usecols=['rf_id', 'ee_name'], low_memory=False)
    ee = ee[ee['rf_id'].isin(valid_rfids2)].copy()
    ee['ee_norm'] = ee['ee_name'].apply(normalize_company)
    ee = ee[ee['ee_norm'].notna()].drop_duplicates('rf_id')[['rf_id', 'ee_norm']]
    print(f'  유효 assignee: {len(ee):,}개')

    # 4. documentid → patent_id
    print('documentid.csv 로드...')
    doc = pd.read_csv(CHAIN_DIR / 'documentid.csv',
                      usecols=['rf_id', 'grant_doc_num'], low_memory=False)
    doc = doc[doc['rf_id'].isin(valid_rfids2)].dropna(subset=['grant_doc_num']).copy()
    doc['patent_norm'] = doc['grant_doc_num'].astype(str).str.strip().apply(normalize_patent)
    doc = doc[doc['patent_norm'].notna()][['rf_id', 'patent_norm']]
    print(f'  유효 patent: {len(doc):,}개')

    # 5. 조인 → chain_df
    chain_df = (
        doc.merge(asgn, on='rf_id').merge(ee, on='rf_id')
        [['patent_norm', 'record_dt', 'ee_norm']]
        .drop_duplicates()
        .sort_values(['patent_norm', 'record_dt'])
        .reset_index(drop=True)
    )
    del doc, asgn, ee
    print(f'chain_df: {len(chain_df):,}행  (unique 특허: {chain_df["patent_norm"].nunique():,}개)')

    # 6. 소유권 구간 생성
    print('소유권 구간 생성 중...')
    INF_DATE = pd.Timestamp('2099-12-31')
    cs = chain_df.sort_values(['patent_norm', 'record_dt']).reset_index(drop=True)
    cs['end_dt'] = cs.groupby('patent_norm')['record_dt'].shift(-1).fillna(INF_DATE)
    intervals_df = cs.rename(columns={'record_dt': 'start_dt'})

    intervals_df.to_parquet(INTERVAL_CACHE, index=False)
    print(f'캐시 저장: {INTERVAL_CACHE}')

print(f'\n소유권 구간: {len(intervals_df):,}행  특허: {intervals_df["patent_norm"].nunique():,}개')

소유권 구간 캐시 로드...


/tmp/ipykernel_3481047/922537180.py:10: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  intervals_df = pd.read_csv(INTERVAL_CACHE, parse_dates=["start_dt", "end_dt"])



소유권 구간: 7,965,741행  특허: 5,193,477개


In [36]:
# Step 7-3. Assignment Chain → date_filed 시점 실소유자로 원고/피고 재매칭 + 통합 통계

def match_to_party(party_str, owner_norm):
    if pd.isna(party_str) or pd.isna(owner_norm):
        return ''
    a_tok = set(str(owner_norm).split())
    for p in party_str.split(' | '):
        p_tok = set(str(p).split())
        if len(a_tok) <= 1:
            if a_tok == p_tok: return p
        else:
            if len(a_tok & p_tok) >= 2: return p
    return ''

# case_pat 기준 + defendants + date_filed 추가
cp = (
    case_pat[case_pat['case_row_id'].isin(case_table['case_row_id'])].copy()
    .merge(case_table[['case_row_id', 'defendants', 'date_filed']], on='case_row_id', how='left')
)

# date_filed 시점에 해당 특허를 소유한 회사 조회 (intervals_df 활용)
lit_iv = intervals_df[intervals_df['patent_norm'].isin(cp['patent_norm'].unique())].copy()
lit_iv['start_dt'] = pd.to_datetime(lit_iv['start_dt'])
lit_iv['end_dt']   = pd.to_datetime(lit_iv['end_dt'])

cp_unmatch = cp[cp['matched_plaintiff'] == ''].copy()
cp_unmatch['date_filed'] = pd.to_datetime(cp_unmatch['date_filed'])

# 특허 × date_filed 기준 실소유자 조인
cp_chain = cp_unmatch.merge(lit_iv, on='patent_norm', how='left')
valid_interval = (
    cp_chain['start_dt'].notna() &
    (cp_chain['start_dt'] <= cp_chain['date_filed']) &
    (cp_chain['date_filed'] < cp_chain['end_dt'])
)
cp_chain = (
    cp_chain[valid_interval]
    .sort_values('start_dt', ascending=False)
    .drop_duplicates(subset=['case_row_id', 'patent_norm'])
    [['case_row_id', 'patent_norm', 'ee_norm']]
    .rename(columns={'ee_norm': 'chain_owner'})
)

# 원래 cp에 chain_owner 붙이기
cp = cp.merge(cp_chain, on=['case_row_id', 'patent_norm'], how='left')

# USPTO 매칭 실패 행에 대해 chain 매칭 시도 (원고 / 피고)
need_chain = cp['matched_plaintiff'] == ''
cp['chain_pla_match'] = ''
cp['chain_def_match'] = ''
cp.loc[need_chain, 'chain_pla_match'] = cp[need_chain].apply(
    lambda r: match_to_party(r['plaintiffs'], r['chain_owner']), axis=1
)
cp.loc[need_chain, 'chain_def_match'] = cp[need_chain].apply(
    lambda r: match_to_party(r['defendants'], r['chain_owner']), axis=1
)

# 특허별 최종 출처 분류
cp['match_src'] = 'unmatched'
cp.loc[cp['matched_plaintiff'] != '', 'match_src'] = 'uspto_plaintiff'
cp.loc[(cp['matched_plaintiff'] == '') & (cp['chain_pla_match'] != ''), 'match_src'] = 'chain_plaintiff'
cp.loc[
    (cp['matched_plaintiff'] == '') & (cp['chain_pla_match'] == '') & (cp['chain_def_match'] != ''),
    'match_src'
] = 'chain_defendant'

# ── 특허 단위 요약 ──────────────────────────────────────────
total_pats = len(cp)
print('=== 특허 단위 매칭 결과 ===')
for src, label in [
    ('uspto_plaintiff',  'USPTO g_assignee   → 원고 매칭'),
    ('chain_plaintiff',  'Assignment Chain   → 원고 매칭'),
    ('chain_defendant',  'Assignment Chain   → 피고 매칭 (소유자=피고)'),
    ('unmatched',        '매칭 실패'),
]:
    n = (cp['match_src'] == src).sum()
    print(f'  {label:38s}: {n:>7,}건 ({n/total_pats:.1%})')

# ── 케이스 단위 × 특허 수 세부 통계 ────────────────────────
cs = cp.groupby('case_row_id').agg(
    total   = ('patent_norm', 'count'),
    n_usp   = ('match_src', lambda x: (x == 'uspto_plaintiff').sum()),
    n_chain = ('match_src', lambda x: (x == 'chain_plaintiff').sum()),
    n_cdef  = ('match_src', lambda x: (x == 'chain_defendant').sum()),
    n_none  = ('match_src', lambda x: (x == 'unmatched').sum()),
).reset_index()

print()
print(f'{"특허수":>5}  {"케이스":>7}  {"USPTO원고":>11}  {"Chain원고":>10}  {"Chain피고":>10}  {"매칭없음":>10}')
buckets = [(n, f'{n}개') for n in range(1, 9)] + [(None, '9개+')]
for val, label in buckets:
    g = cs[cs['total'] == val] if val else cs[cs['total'] >= 9]
    if len(g) == 0: continue
    tot = g['total'].sum()
    def pct(col): return f'{g[col].sum():,}({g[col].sum()/tot:.0%})'
    print(f'{label:>5}  {len(g):>7,}건  {pct("n_usp"):>11}  {pct("n_chain"):>10}  {pct("n_cdef"):>10}  {pct("n_none"):>10}')

tot_all = cs['total'].sum()
def pct_all(col): return f'{cs[col].sum():,}({cs[col].sum()/tot_all:.0%})'
print(f'{"합계":>5}  {len(cs):>7,}건  {pct_all("n_usp"):>11}  {pct_all("n_chain"):>10}  {pct_all("n_cdef"):>10}  {pct_all("n_none"):>10}')

# ── 케이스 단위 요약 ────────────────────────────────────────
cs['all_matched']  = cs['n_none'] == 0
cs['any_matched']  = (cs['n_usp'] + cs['n_chain']) > 0
cs['none_matched'] = (cs['n_usp'] + cs['n_chain']) == 0
print()
print('=== 케이스 단위 최종 요약 (원고 매칭 기준) ===')
print(f'  모든 특허 원고 매칭:   {cs["all_matched"].sum():,}건 ({cs["all_matched"].mean():.1%})')
print(f'  일부라도 원고 매칭:    {(cs["any_matched"] & ~cs["all_matched"]).sum():,}건')
print(f'  원고 매칭 0건:         {cs["none_matched"].sum():,}건 ({cs["none_matched"].mean():.1%})')

=== 특허 단위 매칭 결과 ===
  USPTO g_assignee   → 원고 매칭            :  42,081건 (41.5%)
  Assignment Chain   → 원고 매칭            :   1,447건 (1.4%)
  Assignment Chain   → 피고 매칭 (소유자=피고)   :      98건 (0.1%)
  매칭 실패                                 :  57,666건 (56.9%)

  특허수      케이스      USPTO원고     Chain원고     Chain피고        매칭없음
   1개   25,133건   8,440(34%)     566(2%)      16(0%)  16,111(64%)
   2개    9,060건   7,469(41%)     246(1%)      16(0%)  10,389(57%)
   3개    4,490건   6,008(45%)     148(1%)       8(0%)  7,306(54%)
   4개    2,603건   4,965(48%)      94(1%)      11(0%)  5,342(51%)
   5개    1,609건   3,515(44%)      67(1%)       4(0%)  4,459(55%)
   6개      976건   2,545(43%)      54(1%)      13(0%)  3,244(55%)
   7개      597건   1,951(47%)      22(1%)       8(0%)  2,198(53%)
   8개      314건   1,108(44%)      25(1%)       0(0%)  1,379(55%)
  9개+    1,060건   6,080(45%)     225(2%)      22(0%)  7,238(53%)
   합계   45,842건  42,081(42%)   1,447(1%)      98(0%)  57,666(57%)

=== 케이스 단위 최종 요약 (원고 매칭 기준)

In [24]:
cp['final_plaintiff'] = cp['matched_plaintiff'].where(
    cp['matched_plaintiff'] != '', cp['chain_pla_match']
)
cp['final_plaintiff'] = cp['final_plaintiff'].where(
    cp['final_plaintiff'] != '', cp['plaintiffs'].str.split(' | ').str[0]
)
cp['final_defendant'] = cp['defendants'].str.split(' | ').str[0]

matched_base = cp[['case_row_id', 'date_filed', 'final_plaintiff', 'final_defendant',
                    'patent_norm', 'match_src']].copy()
matched_base.to_csv(OUTPUT_DIR / 'samples/case_patent_matched.csv', index=False)
print(f'저장: {len(matched_base):,}행')
print(matched_base.head(5).to_string())

저장: 101,471행
   case_row_id date_filed final_plaintiff final_defendant patent_norm        match_src
0          727 2003-01-30          magnum        advanced     6457274        unmatched
1          728 2003-02-03  crestliner inc      alumacraft     5108321  uspto_plaintiff
2          729 2003-02-04          taylor              as     6153278        unmatched
3          731 2003-02-12         cardiac               a     5897576        unmatched
4          732 2003-02-21      ecolab inc        anderson     5365690  uspto_plaintiff


In [54]:
none_matched_ids = cs[cs['none_matched']]['case_row_id']
cp_none = cp[cp['case_row_id'].isin(none_matched_ids)].copy()

print(f'원고 매칭 0건 케이스: {len(none_matched_ids):,}건')
print(f'해당 특허 수: {len(cp_none):,}개\n')

no_assignee  = cp_none['assignee_norm'].isna()
no_chain     = cp_none['chain_owner'].isna()

cat1 = no_assignee & no_chain
cat2 = ~no_assignee & no_chain
cat3 = no_assignee & ~no_chain
cat4 = ~no_assignee & ~no_chain

print('=== 원인 분류 ===')
for label, mask in [
    ('USPTO 없음 + Chain 없음 (데이터 자체 부재)', cat1),
    ('USPTO 있음 + Chain 없음 (USPTO 이름 불일치)', cat2),
    ('USPTO 없음 + Chain 있음 (Chain 이름 불일치)', cat3),
    ('USPTO 있음 + Chain 있음 (둘 다 이름 불일치)', cat4),
]:
    print(f'  {label}: {mask.sum():,}개 ({mask.sum()/len(cp_none):.1%})')

# 이름 불일치 샘플 — USPTO
print('\n=== USPTO 이름 불일치 샘플 (cat2) ===')
for _, r in cp_none[cat2].head(10).iterrows():
    print(f'  USPTO소유자: {r["assignee_norm"]}')
    print(f'  원고목록:    {r["plaintiffs"]}')
    print()

# 이름 불일치 샘플 — Chain
print('\n=== Chain 이름 불일치 샘플 (cat3+cat4) ===')
for _, r in cp_none[cat3 | cat4].head(10).iterrows():
    print(f'  Chain소유자: {r["chain_owner"]}')
    print(f'  원고목록:    {r["plaintiffs"]}')
    print()

원고 매칭 0건 케이스: 9,897건
해당 특허 수: 18,619개

=== 원인 분류 ===
  USPTO 없음 + Chain 없음 (데이터 자체 부재): 5,925개 (31.8%)
  USPTO 있음 + Chain 없음 (USPTO 이름 불일치): 1,397개 (7.5%)
  USPTO 없음 + Chain 있음 (Chain 이름 불일치): 2,134개 (11.5%)
  USPTO 있음 + Chain 있음 (둘 다 이름 불일치): 9,163개 (49.2%)

=== USPTO 이름 불일치 샘플 (cat2) ===
  USPTO소유자: minnesota mining and manufacturing co
  원고목록:    3m co | 3m innovative properties co

  USPTO소유자: locker mate corp
  원고목록:    a minnesota corp | lockermate corp

  USPTO소유자: merchant gould
  원고목록:    donaldson co inc

  USPTO소유자: veritec inc
  원고목록:    llc | terminated: 10 25 2006 | vcode holdings inc | vdata | vdata llc

  USPTO소유자: merchant gould
  원고목록:    donaldson co inc

  USPTO소유자: minnesota mining and manufacturing co
  원고목록:    3m co | 3m innovative properties co

  USPTO소유자: veritec inc
  원고목록:    vcode holdings inc | vdata llc

  USPTO소유자: veritec inc
  원고목록:    terminated: 05 05 2006 | vcode holdings inc | vdata llc

  USPTO소유자: minnesota mining and manufacturing co
  원고목록:   

In [58]:
none_matched_ids = cs[cs['none_matched']]['case_row_id']

cross = (
    case_table[case_table['case_row_id'].isin(none_matched_ids)]
    .assign(
        n_patents   = lambda df: df['case_row_id'].map(cs.set_index('case_row_id')['total']),
        n_plaintiffs =  lambda df: df['plaintiffs'].str.split(' | ').str.len()
    )
    .groupby(['n_patents', 'n_plaintiffs'])
    .size()
    .reset_index(name='건수')
)

print(f'원고 매칭 0건 케이스: {len(none_matched_ids):,}건\n')
print(f'{"특허수":>5}  {"원고수":>5}  {"건수":>7}')
print('-' * 22)
for _, r in cross.iterrows():
    print(f'{int(r.n_patents):>4}개  {int(r.n_plaintiffs):>4}명  {int(r["건수"]):>7,}건')

원고 매칭 0건 케이스: 9,897건

  특허수    원고수       건수
----------------------
   1개     1명       17건
   1개     2명    1,320건
   1개     3명    2,077건
   1개     4명      883건
   1개     5명      329건
   1개     6명      309건
   1개     7명      383건
   1개     8명      299건
   1개     9명      146건
   1개    10명      114건
   1개    11명       92건
   1개    12명       64건
   1개    13명       58건
   1개    14명       59건
   1개    15명       46건
   1개    16명       46건
   1개    17명       36건
   1개    18명       35건
   1개    19명       19건
   1개    20명       24건
   1개    21명       14건
   1개    22명        7건
   1개    23명       11건
   1개    24명        4건
   1개    25명        5건
   1개    26명        7건
   1개    27명       11건
   1개    28명        5건
   1개    30명        4건
   1개    31명        4건
   1개    32명        1건
   1개    33명        1건
   1개    35명        1건
   1개    39명        1건
   1개    40명        1건
   1개    41명        1건
   1개    42명        1건
   1개    43명        1건
   1개    49명        1건
   1개    53명        1건
   1개    57명 

In [60]:
import re
memo_pattern = r'consolidated|transferred|assigned|administratively|mdl'

memo_rows = lit_names[
    lit_names['name'].str.contains(memo_pattern, case=False, na=False)
]
print(memo_rows['party_type'].value_counts().head(20))
print()
print(memo_rows[['party_type', 'name']].head(20).to_string())

party_type
Consol Defendant            3283
Consol Plaintiff            1238
Counter Claimant             796
Consol Counter Claimant      481
Consol Counter Defendant     240
Defendant                    134
Counter Defendant             77
Plaintiff                     21
Counter-claimant               6
Consol CounterDft              2
Counter-Claimant               1
Consolidated Defendant         1
Consolidated Plaintiff         1
Consol CounterCla              1
Name: count, dtype: int64

               party_type                                           name
72117           Defendant                       White Consolidated, Ltd.
72118    Counter-Claimant                       White Consolidated, Ltd.
100119          Defendant           Auburn Consolidated Industries, Inc.
100134          Defendant           Auburn Consolidated Industries, Inc.
100163   Counter Claimant           Auburn Consolidated Industries, Inc.
100181   Counter Claimant           Auburn Consolidated Indust

In [61]:
# Consol Plaintiff 중 실제 케이스번호가 name으로 들어온 것
consol_pla = lit_names[lit_names['party_type'] == 'Consol Plaintiff']
print(f'Consol Plaintiff 총: {len(consol_pla):,}건')
print()

# 케이스번호 패턴: "2:14 cv 1173" 같은 형식
case_num_pattern = r'\d+:\d+\s*cv\s*\d+|from\s+\d'
is_case_ref = consol_pla['name'].str.contains(case_num_pattern, case=False, na=False)
print(f'케이스번호 형식 name: {is_case_ref.sum():,}건')
print()
print(consol_pla[is_case_ref]['name'].head(20).to_string())
print()
print('--- 정상 회사명 샘플 ---')
print(consol_pla[~is_case_ref]['name'].head(10).to_string())

Consol Plaintiff 총: 3,626건

케이스번호 형식 name: 805건

385925          Consolidated from 2:12-cv-515
406123    Consolidated Civil Action 2:13cv948
407128         Consolidated from 2:13-cv-1050
407130         Consolidated from 2:13-cv-1050
407132         Consolidated from 2:13-cv-1051
407134         Consolidated from 2:13-cv-1051
407136         Consolidated from 2:13-cv-1052
407138         Consolidated from 2:13-cv-1052
407140         Consolidated from 2:13-cv-1053
407142         Consolidated from 2:13-cv-1053
408648         Consolidated from 2:13-cv-1152
408650         Consolidated from 2:13-cv-1153
408652           Consolidated from 2:14-cv-40
408654           Consolidated from 2:14-cv-42
408656           Consolidated from 2:14-cv-43
408658            Consolidated from 2:14-cv44
412807     Consolidated Civil Action 2:15cv25
413377    Consolidated Civil Action 2:14cv209
413379    Consolidated Civil Action 2:14cv211
413382    Consolidated Civil Action 2:14cv136

--- 정상 회사명 샘플 ---
73464       

In [59]:
none_pat1 = (
    case_table[case_table['case_row_id'].isin(none_matched_ids)]
    .assign(n_patents = lambda df: df['case_row_id'].map(cs.set_index('case_row_id')['total']))
    .query('n_patents == 1')
    [['case_row_id', 'date_filed', 'plaintiffs', 'defendants']]
    .merge(
        cp[cp['match_src'] == 'unmatched'][['case_row_id', 'patent_norm', 'assignee_norm', 'chain_owner']],
        on='case_row_id', how='left'
    )
    .sort_values('plaintiffs', key=lambda x: x.str.split(' | ').str.len(), ascending=False)
    .reset_index(drop=True)
)

pd.set_option('display.max_colwidth', 80)
none_pat1[['case_row_id', 'patent_norm', 'assignee_norm', 'chain_owner', 'plaintiffs']].head(30)

,case_row_id,patent_norm,assignee_norm,chain_owner,plaintiffs
0,44909,5751346,dozier financial corp,dozier financial corp,consolidated from 2:14 cv 1173 | consolidated from 2:14 cv 1174 | consolidat...
1,45883,8132720,NaN,NaN,consolidated from case 2:15 cv 480 | consolidated from case 2:15 cv 481 | co...
2,46757,6310864,paradyne corp,paradyne corp,consolidated from 2:15 cv 1303 | consolidated from 2:15 cv 1316 | consolidat...
3,45969,6442261,avaya technology llc,trident ip solutions llc,consolidated civil action 2:15cv563 | consolidated civil action 2:15cv564 | ...
4,43744,6185678,the trustees of the university of pennsylvania,university of pennsylvania,consolidated from case 2:14 cv 867 unconsolidated per order # 88 | consolida...
5,11517,6165577,NaN,NaN,berger s table pad factory inc | berger table pads inc | david berger | fact...
6,44753,7479899,legalview assets ltd,horstemeyer scott andrew,consolidated civil action 2:14cv1059 | consolidated civil action 2:14cv1060 ...
7,52721,5704202,NaN,NaN,a corp organized under laws of the state of illinois with principal place of...
8,35707,5930474,z land llc,labranza est,a corp organized and existing under the laws of texas | a delaware ltd liabi...
9,15500,7301156,NaN,NaN,alice w enge | elizabeth dill | harald enge | in her capacity as executrix o...


In [62]:
# 원고 매칭 0% 케이스 → 수동 검토용 CSV 생성
none_matched_ids = cs[cs['none_matched']]['case_row_id']

# 1. 해당 케이스의 특허 정보
base = cp[cp['case_row_id'].isin(none_matched_ids)][
    ['case_row_id', 'patent_norm', 'assignee_norm', 'chain_owner',
     'plaintiffs', 'defendants', 'date_filed']
].copy()

# 2. names.csv 원본 (raw) — 해당 케이스의 모든 당사자
raw_names = (
    lit_names[lit_names['case_row_id'].isin(none_matched_ids)]
    [['case_row_id', 'party_type', 'name']]
    .groupby('case_row_id')
    .apply(lambda df: ' | '.join(
        f"[{r.party_type}] {r.name}" for _, r in df.iterrows()
    ))
    .reset_index(name='all_parties_raw')
)

# 3. 합치기
result = base.merge(raw_names, on='case_row_id', how='left')

out_path = OUTPUT_DIR / 'samples/0percent_cases.csv'
result.to_csv(out_path, index=False)
print(f'저장: {out_path}')
print(f'행 수: {len(result):,}  케이스: {result["case_row_id"].nunique():,}')
result.head(3)

저장: ../data/samples/0percent_cases.csv
행 수: 18,619  케이스: 9,897


/tmp/ipykernel_2978082/1771966252.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: ' | '.join(


,case_row_id,patent_norm,assignee_norm,chain_owner,plaintiffs,defendants,date_filed,all_parties_raw
0,742,6411942,fujitsu ltd,fujitsu ltd,a delaware corp | firepond inc,banter inc,2003-04-23,[Plaintiff] 4958 | [Plaintiff] 4959 | [Defendant] 4960 | [Defendant] 4961
1,745,4916910,lancer corp,lancer partnership ltd,imi cornelius inc,lancer corp | lancer partnership ltd,2003-05-02,[Plaintiff] 4971 | [Defendant] 4972 | [Defendant] 4973 | [Counter Claimant] ...
2,746,6285752,NaN,NaN,universal recovery systems inc,marcom technologies inc,2003-05-05,[Plaintiff] 4978 | [Defendant] 4979


In [64]:
raw_names = (
    names[names['case_row_id'].isin(none_matched_ids)]
    [['case_row_id', 'party_type', 'name']]
    .groupby('case_row_id')
    .apply(lambda df: ' | '.join(
        f"[{r.party_type}] {r['name']}" for _, r in df.iterrows()
    ))
    .reset_index(name='all_parties_raw')
)
base = cp[cp['case_row_id'].isin(none_matched_ids)][
    ['case_row_id', 'patent_norm', 'assignee_norm', 'chain_owner',
     'plaintiffs', 'defendants', 'date_filed']
].copy()

result = base.merge(raw_names, on='case_row_id', how='left')
result.to_csv(OUTPUT_DIR / 'samples/0percent_cases_raw.csv', index=False)
print(f'저장 완료: {len(result):,}행')
result.head(3)

저장 완료: 18,619행


/tmp/ipykernel_2978082/1584003527.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: ' | '.join(


,case_row_id,patent_norm,assignee_norm,chain_owner,plaintiffs,defendants,date_filed,all_parties_raw
0,742,6411942,fujitsu ltd,fujitsu ltd,a delaware corp | firepond inc,banter inc,2003-04-23,"[Plaintiff] Firepond, Inc. | [Plaintiff] a Delaware corporation | [Defendant..."
1,745,4916910,lancer corp,lancer partnership ltd,imi cornelius inc,lancer corp | lancer partnership ltd,2003-05-02,[Plaintiff] IMI Cornelius Inc. | [Defendant] Lancer Corporation | [Defendant...
2,746,6285752,NaN,NaN,universal recovery systems inc,marcom technologies inc,2003-05-05,"[Plaintiff] Universal Recovery Systems, Inc. | [Defendant] MarCom Technologi..."


In [ ]:
names[names['case_row_id'] == 42][['party_type', 'name']]

In [ ]:
# Cell 36
# 트리플 생성 전 피고 품질 확인 → CSV 저장

DEF_TYPES = {
    'Defendant', 'Counter Defendant', 'Consol Defendant',
    'Counter-defendant', 'ThirdParty Defendant', 'Cross Defendant',
    'defendant', 'Counter-Defendant', 'Consol Counter Defendant',
    'Third Party Defendant', 'Intervenor Defendant', 'Defendant Consolidated',
    'counter-defendant', '3rd party defendant', 'Consolidated Defendant',
    'Counterclaim Defendant', 'Cross-defendant', '3rd Pty Defendant',
    'third-party defendant', 'Third-Party Defendant',
}

# 피고 추출 및 정규화
def_names = names[names['party_type'].isin(DEF_TYPES)][['case_row_id', 'name']].copy()
def_names['name_norm'] = def_names['name'].apply(normalize_company)

total = len(def_names)
noise = def_names['name_norm'].isna().sum()
valid = total - noise
print(f'전체: {total:,}  노이즈: {noise:,} ({noise/total:.1%})  유효: {valid:,} ({valid/total:.1%})')

# 고유 이름별 빈도 + 대표 raw 이름
freq = (
    def_names.dropna(subset=['name_norm'])
    .groupby('name_norm')
    .agg(count=('name', 'count'), name_raw=('name', lambda x: x.value_counts().index[0]))
    .reset_index()
    .sort_values('count', ascending=False)
)

# USPTO assignee 로딩
print('USPTO assignee 로딩 중...')
asgn_list = []
for chunk in pd.read_csv(
    USPTO_DIR / 'g_assignee_disambiguated.tsv', sep='	',
    usecols=['disambig_assignee_organization'], chunksize=500_000
):
    normed = (
        chunk['disambig_assignee_organization']
        .dropna()
        .drop_duplicates()
    )
    asgn_list.append(normed)

asgn_raw_df = pd.concat(asgn_list).drop_duplicates().reset_index(drop=True)
asgn_raw_df = asgn_raw_df.to_frame(name='asgn_raw')
asgn_raw_df['asgn_norm'] = asgn_raw_df['asgn_raw'].apply(normalize_company)
asgn_raw_df = asgn_raw_df.dropna(subset=['asgn_norm']).drop_duplicates('asgn_norm')
asgn_norm_to_raw = dict(zip(asgn_raw_df['asgn_norm'], asgn_raw_df['asgn_raw']))
asgn_norm_set = set(asgn_norm_to_raw.keys())
print(f'USPTO 고유 기관: {len(asgn_norm_set):,}')

# 매칭 여부 + USPTO 원본 표기 찾기
freq['uspto_match'] = freq['name_norm'].isin(asgn_norm_set)
freq['uspto_name']  = freq['name_norm'].map(asgn_norm_to_raw)

# 미매칭 중 토큰 겹침으로 가장 가까운 USPTO 이름 찾기
print('미매칭 후보 검색 중...')
asgn_tokens = {n: set(n.split()) for n in asgn_norm_set}

def find_closest(norm_name):
    if pd.isna(norm_name):
        return None
    tokens = set(norm_name.split())
    if len(tokens) == 0:
        return None
    best_score, best_name = 0, None
    for asgn_n, asgn_t in asgn_tokens.items():
        score = len(tokens & asgn_t) / max(len(tokens | asgn_t), 1)
        if score > best_score:
            best_score, best_name = score, asgn_n
    return asgn_norm_to_raw.get(best_name) if best_score >= 0.6 else None

unmatched = freq[~freq['uspto_match']].copy()
print(f'미매칭: {len(unmatched):,}건 (행 기준: {unmatched['count'].sum():,}건)')
unmatched['uspto_closest'] = unmatched['name_norm'].apply(find_closest)
freq.loc[~freq['uspto_match'], 'uspto_closest'] = unmatched['uspto_closest'].values

# 전체 통계 출력
matched   = freq['uspto_match'].sum()
unmatched_n = (~freq['uspto_match']).sum()
print()
print(f'=== 고유 이름 기준 ===')
print(f'  USPTO 매칭:  {matched:,} ({matched/len(freq):.1%})')
print(f'  USPTO 미매칭: {unmatched_n:,} ({unmatched_n/len(freq):.1%})')
print()
print(f'=== 행(등장 횟수) 기준 ===')
m_cnt  = freq[freq['uspto_match']]['count'].sum()
um_cnt = freq[~freq['uspto_match']]['count'].sum()
print(f'  USPTO 매칭:  {m_cnt:,} ({m_cnt/(m_cnt+um_cnt):.1%})')
print(f'  USPTO 미매칭: {um_cnt:,} ({um_cnt/(m_cnt+um_cnt):.1%})')

# CSV 저장
out_path = OUTPUT_DIR / 'samples/defendant_quality.csv'
freq[['name_norm', 'name_raw', 'count', 'uspto_match', 'uspto_name', 'uspto_closest']].to_csv(out_path, index=False)
print('CSV 저장:', out_path, f'({len(freq):,}행)')

freq[['name_norm', 'name_raw', 'count', 'uspto_match', 'uspto_name', 'uspto_closest']].head(30)
